In [1]:
!apt-get update -qq
!apt-get install -y poppler-utils tesseract-ocr -qq

!pip install -q \
    streamlit \
    langgraph \
    langchain-google-genai \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    faiss-cpu \
    pdfplumber \
    pdf2image \
    pillow \
    pydantic

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
!apt-get update -qq
!apt-get install -y poppler-utils tesseract-ocr -qq

!pip install -q \
    streamlit \
    langgraph \
    langchain-google-genai \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    faiss-cpu \
    pdfplumber \
    pdf2image \
    pillow \
    pydantic

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
!apt-get update -qq
!apt-get install -y poppler-utils tesseract-ocr -qq

!pip install -q \
    streamlit \
    langgraph \
    langchain-google-genai \
    langchain-core \
    langchain-community \
    langchain-text-splitters \
    faiss-cpu \
    pdfplumber \
    pdf2image \
    pillow \
    pydantic \
    pyngrok

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

(Reading database ... 118456 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.8.2) over (2026.8.2) ...
Setting up cloudflared (2026.8.2) ...
Processing triggers for man-db (2.10.2-1) ...


In [5]:
import subprocess
import time
import re

# 1. Kill any existing Streamlit or cloudflared instances
!pkill -f streamlit
!pkill -f cloudflared

# 2. Start Streamlit in the background on port 8501
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])

# Give Streamlit a moment to start up
time.sleep(3)

# 3. Start Cloudflare Tunnel
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

# 4. Extract and print the public trycloudflare.com URL
url_found = False
for line in iter(tunnel.stderr.readline, ''):
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-0-]+\.trycloudflare\.com', line)
        if match:
            print("=========================================================")
            print(f"🚀 YOUR LIVE APP URL:\n{match.group(0)}")
            print("=========================================================")
            url_found = True
            break

if not url_found:
    print("Could not retrieve tunnel URL. Please re-run this cell.")

🚀 YOUR LIVE APP URL:
https://transformation-base-posing-constitutes.trycloudflare.com


In [6]:
%%writefile app.py
import os
import base64
import tempfile
from io import BytesIO
from typing import TypedDict, Optional

import streamlit as st
from PIL import Image
import pdfplumber
from pdf2image import convert_from_path

from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END

# Page Setup
st.set_page_config(page_title="Agentic PDF OCR + RAG", page_icon="📄", layout="wide")
st.title("📄 Agentic PDF OCR & RAG Assistant")

# Sidebar Setup
with st.sidebar:
    st.header("Configuration")
    gemini_api_key = st.text_input("Google Gemini API Key", type="password")
    # Updated to active current Gemini models
    selected_model = st.selectbox("LLM Model", ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-2.5-pro"])

if not gemini_api_key or len(gemini_api_key.strip()) < 10:
    st.info("👈 Enter a valid Gemini API Key in the sidebar to start.")
    st.stop()

os.environ["GOOGLE_API_KEY"] = gemini_api_key.strip()

# State Schema
class AgentState(TypedDict):
    file_path: str
    extracted_text: str
    is_scanned: bool
    page_count: int
    final_markdown: str
    vectorstore: Optional[object]

@st.cache_resource
def get_compiled_graph(model_name: str, api_key: str):
    llm = ChatGoogleGenerativeAI(
        model=model_name,
        google_api_key=api_key,
        temperature=0
    )

    def extract_direct_text(state: AgentState) -> AgentState:
        text = ""
        page_count = 0
        try:
            with pdfplumber.open(state["file_path"]) as pdf:
                page_count = len(pdf.pages)
                for page in pdf.pages:
                    extracted = page.extract_text()
                    if extracted: text += extracted + "\n"
        except Exception:
            return {**state, "extracted_text": "", "is_scanned": True, "page_count": 0}

        is_scanned = (len(text.strip()) / max(page_count, 1)) < 50
        return {**state, "extracted_text": text, "is_scanned": is_scanned, "page_count": page_count}

    def vision_ocr_fallback(state: AgentState) -> AgentState:
        images = convert_from_path(state["file_path"], first_page=1, last_page=3)
        ocr_results = []
        for i, img in enumerate(images):
            buffered = BytesIO()
            img.save(buffered, format="JPEG")
            img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
            msg = HumanMessage(
                content=[
                    {"type": "text", "text": "Extract all text cleanly."},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
                ]
            )
            ocr_results.append(f"--- Page {i+1} ---\n" + llm.invoke([msg]).content)
        return {**state, "extracted_text": "\n\n".join(ocr_results)}

    def refine_and_format(state: AgentState) -> AgentState:
        prompt = f"Format this OCR text cleanly into Markdown:\n\n{state['extracted_text']}"
        res = llm.invoke(prompt)
        return {**state, "final_markdown": res.content}

    def build_rag_index(state: AgentState) -> AgentState:
        text = state["final_markdown"]
        if not text.strip(): return {**state, "vectorstore": None}
        docs = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).create_documents([text])
        embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=api_key)
        return {**state, "vectorstore": FAISS.from_documents(docs, embeddings)}

    workflow = StateGraph(AgentState)
    workflow.add_node("direct", extract_direct_text)
    workflow.add_node("vision", vision_ocr_fallback)
    workflow.add_node("refine", refine_and_format)
    workflow.add_node("rag", build_rag_index)

    workflow.add_edge(START, "direct")
    workflow.add_conditional_edges("direct", lambda s: "vision" if s["is_scanned"] else "refine")
    workflow.add_edge("vision", "refine")
    workflow.add_edge("refine", "rag")
    workflow.add_edge("rag", END)

    return workflow.compile(), llm

agent_app, llm_instance = get_compiled_graph(selected_model, gemini_api_key)

uploaded_file = st.file_uploader("Upload PDF", type=["pdf"])

if uploaded_file:
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(uploaded_file.read())
        tmp_path = tmp.name

    if "agent_result" not in st.session_state or st.session_state.get("file_name") != uploaded_file.name:
        with st.spinner("Processing document..."):
            result = agent_app.invoke({"file_path": tmp_path, "extracted_text": "", "is_scanned": False, "page_count": 0, "final_markdown": "", "vectorstore": None})
            st.session_state["agent_result"] = result
            st.session_state["file_name"] = uploaded_file.name

    os.remove(tmp_path)
    result = st.session_state["agent_result"]

    st.subheader("Summary")
    st.write(f"**Mode:** {'Vision OCR' if result['is_scanned'] else 'Direct Extract'}")

    tab1, tab2 = st.tabs(["📄 OCR Result", "💬 RAG Question Answering"])
    with tab1:
        st.markdown(result["final_markdown"])

    with tab2:
        vectorstore = result.get("vectorstore")
        if vectorstore:
            query = st.text_input("Ask a question about this PDF:")
            if query:
                retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
                docs = retriever.invoke(query)
                ctx = "\n\n".join([d.page_content for d in docs])
                ans = llm_instance.invoke(f"Context:\n{ctx}\n\nQuestion: {query}")
                st.markdown("### Answer")
                st.write(ans.content)

Overwriting app.py


In [7]:
import os
import subprocess
from google.colab import output

# 1. Kill any existing Streamlit processes
!pkill streamlit

# 2. Start Streamlit server in the background
process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"])

# 3. Generate Colab's native public proxy URL
proxy_url = output.eval_js("google.colab.kernel.proxyPort(8501)")

print("==========================================================================")
print(f"🚀 YOUR LIVE APP IS READY HERE:\n\n{proxy_url}")
print("==========================================================================")

🚀 YOUR LIVE APP IS READY HERE:

https://8501-m-s-kkb-use1d1-1sfqdh4fo3kan-d.us-east1-1.prod.colab.dev


In [8]:
import subprocess
import time
import re

# 1. Kill existing processes
!pkill -f streamlit
!pkill -f cloudflared

# 2. Launch Streamlit WITH CORS & XSRF DISABLED (Fixes infinite loading!)
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

time.sleep(3)

# 3. Start Cloudflare Tunnel
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

# 4. Get the URL
for line in iter(tunnel.stderr.readline, ''):
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print("=========================================================")
            print(f"🚀 YOUR FIXED LIVE APP URL:\n{match.group(0)}")
            print("=========================================================")
            break

🚀 YOUR FIXED LIVE APP URL:
https://ethics-kim-moon-dod.trycloudflare.com


In [9]:
%%writefile app.py
import os
import base64
import tempfile
from io import BytesIO
from typing import TypedDict, Optional

import streamlit as st
from PIL import Image
import pdfplumber
from pdf2image import convert_from_path

from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END

# Page Setup
st.set_page_config(page_title="Agentic PDF OCR + RAG", page_icon="📄", layout="wide")
st.title("📄 Agentic PDF OCR & RAG Assistant")

# Sidebar Setup
with st.sidebar:
    st.header("Configuration")
    gemini_api_key = st.text_input("Google Gemini API Key", type="password")
    selected_model = st.selectbox("LLM Model", ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-2.5-pro"])

if not gemini_api_key or len(gemini_api_key.strip()) < 10:
    st.info("👈 Enter a valid Gemini API Key in the sidebar to start.")
    st.stop()

os.environ["GOOGLE_API_KEY"] = gemini_api_key.strip()

# State Schema
class AgentState(TypedDict):
    file_path: str
    extracted_text: str
    is_scanned: bool
    page_count: int
    final_markdown: str
    vectorstore: Optional[object]

@st.cache_resource
def get_compiled_graph(model_name: str, api_key: str):
    llm = ChatGoogleGenerativeAI(
        model=model_name,
        google_api_key=api_key,
        temperature=0
    )

    def extract_direct_text(state: AgentState) -> AgentState:
        text = ""
        page_count = 0
        try:
            with pdfplumber.open(state["file_path"]) as pdf:
                page_count = len(pdf.pages)
                for page in pdf.pages:
                    extracted = page.extract_text()
                    if extracted: text += extracted + "\n"
        except Exception:
            return {**state, "extracted_text": "", "is_scanned": True, "page_count": 0}

        is_scanned = (len(text.strip()) / max(page_count, 1)) < 50
        return {**state, "extracted_text": text, "is_scanned": is_scanned, "page_count": page_count}

    def vision_ocr_fallback(state: AgentState) -> AgentState:
        images = convert_from_path(state["file_path"], first_page=1, last_page=3)
        ocr_results = []
        for i, img in enumerate(images):
            buffered = BytesIO()
            img.save(buffered, format="JPEG")
            img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
            msg = HumanMessage(
                content=[
                    {"type": "text", "text": "Extract all text cleanly."},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
                ]
            )
            ocr_results.append(f"--- Page {i+1} ---\n" + llm.invoke([msg]).content)
        return {**state, "extracted_text": "\n\n".join(ocr_results)}

    def refine_and_format(state: AgentState) -> AgentState:
        prompt = f"Format this OCR text cleanly into Markdown:\n\n{state['extracted_text']}"
        res = llm.invoke(prompt)
        return {**state, "final_markdown": res.content}

    def build_rag_index(state: AgentState) -> AgentState:
        text = state["final_markdown"]
        if not text.strip(): return {**state, "vectorstore": None}
        docs = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).create_documents([text])

        # FIX: Updated to supported embedding model 'text-embedding-004'
        embeddings = GoogleGenerativeAIEmbeddings(model="text-embedding-004", google_api_key=api_key)
        return {**state, "vectorstore": FAISS.from_documents(docs, embeddings)}

    workflow = StateGraph(AgentState)
    workflow.add_node("direct", extract_direct_text)
    workflow.add_node("vision", vision_ocr_fallback)
    workflow.add_node("refine", refine_and_format)
    workflow.add_node("rag", build_rag_index)

    workflow.add_edge(START, "direct")
    workflow.add_conditional_edges("direct", lambda s: "vision" if s["is_scanned"] else "refine")
    workflow.add_edge("vision", "refine")
    workflow.add_edge("refine", "rag")
    workflow.add_edge("rag", END)

    return workflow.compile(), llm

agent_app, llm_instance = get_compiled_graph(selected_model, gemini_api_key)

uploaded_file = st.file_uploader("Upload PDF", type=["pdf"])

if uploaded_file:
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(uploaded_file.read())
        tmp_path = tmp.name

    if "agent_result" not in st.session_state or st.session_state.get("file_name") != uploaded_file.name:
        with st.spinner("Processing document..."):
            result = agent_app.invoke({"file_path": tmp_path, "extracted_text": "", "is_scanned": False, "page_count": 0, "final_markdown": "", "vectorstore": None})
            st.session_state["agent_result"] = result
            st.session_state["file_name"] = uploaded_file.name

    os.remove(tmp_path)
    result = st.session_state["agent_result"]

    st.subheader("Summary")
    st.write(f"**Mode:** {'Vision OCR' if result['is_scanned'] else 'Direct Extract'}")

    tab1, tab2 = st.tabs(["📄 OCR Result", "💬 RAG Question Answering"])
    with tab1:
        st.markdown(result["final_markdown"])

    with tab2:
        vectorstore = result.get("vectorstore")
        if vectorstore:
            query = st.text_input("Ask a question about this PDF:")
            if query:
                retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
                docs = retriever.invoke(query)
                ctx = "\n\n".join([d.page_content for d in docs])
                ans = llm_instance.invoke(f"Context:\n{ctx}\n\nQuestion: {query}")
                st.markdown("### Answer")
                st.write(ans.content)

Overwriting app.py


In [10]:
import subprocess, time, re

!pkill -f streamlit
!pkill -f cloudflared

subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

time.sleep(3)

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

for line in iter(tunnel.stderr.readline, ''):
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print(f"🚀 LIVE APP URL: {match.group(0)}")
            break

🚀 LIVE APP URL: https://ozone-mentioned-talking-applied.trycloudflare.com


In [11]:
%%writefile app.py
import os
import base64
import tempfile
from io import BytesIO
from typing import TypedDict, Optional

import streamlit as st
from PIL import Image
import pdfplumber
from pdf2image import convert_from_path

from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END

# Page Setup
st.set_page_config(page_title="Agentic PDF OCR + RAG", page_icon="📄", layout="wide")
st.title("📄 Agentic PDF OCR & RAG Assistant")

# Sidebar Setup
with st.sidebar:
    st.header("Configuration")
    gemini_api_key = st.text_input("Google Gemini API Key", type="password")
    selected_model = st.selectbox("LLM Model", ["gemini-2.5-flash", "gemini-2.0-flash", "gemini-2.5-pro"])

if not gemini_api_key or len(gemini_api_key.strip()) < 10:
    st.info("👈 Enter a valid Gemini API Key in the sidebar to start.")
    st.stop()

os.environ["GOOGLE_API_KEY"] = gemini_api_key.strip()

# State Schema
class AgentState(TypedDict):
    file_path: str
    extracted_text: str
    is_scanned: bool
    page_count: int
    final_markdown: str
    vectorstore: Optional[object]

@st.cache_resource
def get_compiled_graph(model_name: str, api_key: str):
    llm = ChatGoogleGenerativeAI(
        model=model_name,
        google_api_key=api_key,
        temperature=0
    )

    def extract_direct_text(state: AgentState) -> AgentState:
        text = ""
        page_count = 0
        try:
            with pdfplumber.open(state["file_path"]) as pdf:
                page_count = len(pdf.pages)
                for page in pdf.pages:
                    extracted = page.extract_text()
                    if extracted: text += extracted + "\n"
        except Exception:
            return {**state, "extracted_text": "", "is_scanned": True, "page_count": 0}

        is_scanned = (len(text.strip()) / max(page_count, 1)) < 50
        return {**state, "extracted_text": text, "is_scanned": is_scanned, "page_count": page_count}

    def vision_ocr_fallback(state: AgentState) -> AgentState:
        images = convert_from_path(state["file_path"], first_page=1, last_page=3)
        ocr_results = []
        for i, img in enumerate(images):
            buffered = BytesIO()
            img.save(buffered, format="JPEG")
            img_b64 = base64.b64encode(buffered.getvalue()).decode("utf-8")
            msg = HumanMessage(
                content=[
                    {"type": "text", "text": "Extract all text cleanly."},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{img_b64}"}}
                ]
            )
            ocr_results.append(f"--- Page {i+1} ---\n" + llm.invoke([msg]).content)
        return {**state, "extracted_text": "\n\n".join(ocr_results)}

    def refine_and_format(state: AgentState) -> AgentState:
        prompt = f"Format this OCR text cleanly into Markdown:\n\n{state['extracted_text']}"
        res = llm.invoke(prompt)
        return {**state, "final_markdown": res.content}

    def build_rag_index(state: AgentState) -> AgentState:
        text = state["final_markdown"]
        if not text.strip(): return {**state, "vectorstore": None}
        docs = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50).create_documents([text])

        # UPDATED: Supported current embedding model string
        embeddings = GoogleGenerativeAIEmbeddings(
            model="gemini-embedding-2-preview",
            google_api_key=api_key
        )
        return {**state, "vectorstore": FAISS.from_documents(docs, embeddings)}

    workflow = StateGraph(AgentState)
    workflow.add_node("direct", extract_direct_text)
    workflow.add_node("vision", vision_ocr_fallback)
    workflow.add_node("refine", refine_and_format)
    workflow.add_node("rag", build_rag_index)

    workflow.add_edge(START, "direct")
    workflow.add_conditional_edges("direct", lambda s: "vision" if s["is_scanned"] else "refine")
    workflow.add_edge("vision", "refine")
    workflow.add_edge("refine", "rag")
    workflow.add_edge("rag", END)

    return workflow.compile(), llm

agent_app, llm_instance = get_compiled_graph(selected_model, gemini_api_key)

uploaded_file = st.file_uploader("Upload PDF", type=["pdf"])

if uploaded_file:
    with tempfile.NamedTemporaryFile(delete=False, suffix=".pdf") as tmp:
        tmp.write(uploaded_file.read())
        tmp_path = tmp.name

    if "agent_result" not in st.session_state or st.session_state.get("file_name") != uploaded_file.name:
        with st.spinner("Processing document..."):
            result = agent_app.invoke({"file_path": tmp_path, "extracted_text": "", "is_scanned": False, "page_count": 0, "final_markdown": "", "vectorstore": None})
            st.session_state["agent_result"] = result
            st.session_state["file_name"] = uploaded_file.name

    os.remove(tmp_path)
    result = st.session_state["agent_result"]

    st.subheader("Summary")
    st.write(f"**Mode:** {'Vision OCR' if result['is_scanned'] else 'Direct Extract'}")

    tab1, tab2 = st.tabs(["📄 OCR Result", "💬 RAG Question Answering"])
    with tab1:
        st.markdown(result["final_markdown"])

    with tab2:
        vectorstore = result.get("vectorstore")
        if vectorstore:
            query = st.text_input("Ask a question about this PDF:")
            if query:
                retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
                docs = retriever.invoke(query)
                ctx = "\n\n".join([d.page_content for d in docs])
                ans = llm_instance.invoke(f"Context:\n{ctx}\n\nQuestion: {query}")
                st.markdown("### Answer")
                st.write(ans.content)

Overwriting app.py


In [12]:
import subprocess, time, re

!pkill -f streamlit
!pkill -f cloudflared

subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
])

time.sleep(3)

tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

for line in iter(tunnel.stderr.readline, ''):
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print(f"🚀 LIVE APP URL: {match.group(0)}")
            break

🚀 LIVE APP URL: https://evans-petition-covering-bush.trycloudflare.com
